# Quorift — Grounded Answer Generation Prototype

This notebook executes answer generation for 6 approved development questions from the Quorift banking assistant benchmark using `Qwen/Qwen3-4B-Instruct-2507` on a free Google Colab GPU (T4 or higher).

### Workflow:
1. **Runtime Verification**: Verify CUDA GPU hardware accelerator is available.
2. **Dependency Setup**: Install Hugging Face `transformers` and `accelerate`.
3. **Code Import**: Clone Quorift repository and import trusted generation logic.
4. **Input Upload**: Upload `requests.jsonl` produced by local retrieval on Windows.
5. **Model Loading**: Load `Qwen/Qwen3-4B-Instruct-2507` in FP16 on CUDA.
6. **Grounded Generation**: Generate answers using fixed greedy parameters, strict citation validation, and safe refusal.
7. **Export Results**: Download `generation_results.jsonl`.

In [ ]:
# Cell 1 — Runtime checks
import torch
import sys

if not torch.cuda.is_available():
    raise SystemError(
        "CUDA GPU is not available! Please enable GPU hardware accelerator in Colab: "
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[OK] CUDA GPU available: {gpu_name}")
print(f"[OK] VRAM Capacity: {total_mem_gb:.2f} GB")

In [ ]:
# Cell 2 — Install dependencies
!pip install --quiet --upgrade "transformers>=4.51.0,<5.0.0" "accelerate>=0.26.0"

import transformers
from packaging import version

installed_ver = transformers.__version__
print(f"[OK] Installed transformers version: {installed_ver}")
if version.parse(installed_ver) < version.parse("4.51.0"):
    raise RuntimeError(f"Transformers version {installed_ver} is older than required 4.51.0 for Qwen3.")
print("[OK] Dependencies installed and version requirement verified.")

In [ ]:
# Cell 3 — Access project generation code
import os
import sys

if not os.path.exists("Quorift"):
    !git clone https://github.com/nayanojwal0810/Quorift.git

src_path = os.path.abspath("Quorift/src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import quorift.generation as gen
print("[OK] quorift.generation module imported successfully from repository.")

In [ ]:
# Cell 4 — Upload generation input requests
import json
from google.colab import files

print("Please select and upload 'requests.jsonl' generated by scripts.prepare_generation_inputs:")
uploaded = files.upload()

input_filename = "requests.jsonl"
if input_filename not in uploaded:
    uploaded_keys = list(uploaded.keys())
    if len(uploaded_keys) == 1 and uploaded_keys[0].endswith(".jsonl"):
        input_filename = uploaded_keys[0]
    else:
        raise FileNotFoundError(f"Could not find requests.jsonl in uploaded files: {uploaded_keys}")

requests = []
with open(input_filename, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            requests.append(json.loads(line))

expected_qids = ["Q-001", "Q-018", "Q-025", "Q-033", "Q-037", "Q-041"]
actual_qids = [r["question_id"] for r in requests]
if actual_qids != expected_qids:
    raise ValueError(f"Expected exact question IDs {expected_qids}, got: {actual_qids}")

print(f"[OK] Successfully loaded and validated {len(requests)} input requests: {actual_qids}")

In [ ]:
# Cell 5 — Load Qwen/Qwen3-4B-Instruct-2507 in FP16 on CUDA
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-4B-Instruct-2507"
print(f"Loading {model_id} in float16 onto CUDA GPU...")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True,
)
model.eval()
print(f"[OK] {model_id} loaded successfully on {gpu_name}.")

In [ ]:
# Cell 6 — Grounded Answer Generation and Citation Validation
import time

registry_path = "Quorift/data/metadata/document_registry.json"
with open(registry_path, "r", encoding="utf-8") as f:
    registry = {r["document_id"]: r for r in json.load(f)}

MAX_INPUT_TOKENS = 3072
MAX_NEW_TOKENS = 320

results = []

for req in requests:
    qid = req["question_id"]
    q_text = req["question"]
    role = req["role"]
    chunks = req["retrieved_chunks"]

    print(f"\nProcessing {qid} (Role: {role})...")
    t0 = time.perf_counter()

    # 1. Validate request
    gen.validate_generation_request(req, registry)

    # 2. Check empty evidence
    if len(chunks) == 0:
        empty_resp = gen.generate_empty_evidence_response()
        gen_time = time.perf_counter() - t0
        results.append({
            "question_id": qid,
            "model_id": model_id,
            "raw_model_output": json.dumps(empty_resp),
            "parsed_status": empty_resp["status"],
            "parsed_answer": empty_resp["answer"],
            "validated_citations": empty_resp["citations"],
            "validation_passed": True,
            "validation_error": None,
            "generation_time_seconds": gen_time,
        })
        print(f"  [Empty Evidence Refusal] Answer: {empty_resp['answer']}")
        continue

    # 3. Build grounded prompt
    messages = gen.build_grounded_prompt(q_text, chunks)

    # 4. Tokenize & check length constraint
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    input_token_count = inputs["input_ids"].shape[1]
    if input_token_count > MAX_INPUT_TOKENS:
        raise ValueError(f"Prompt token length {input_token_count} exceeds maximum {MAX_INPUT_TOKENS}")

    # 5. Generate under torch.inference_mode()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_time = time.perf_counter() - t0

    # Decode newly generated tokens only
    new_tokens = output_ids[0][input_token_count:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # 6. Parse and validate
    try:
        parsed = gen.parse_generation_response(raw_output)
        validated = gen.validate_generation_output(parsed, chunks, registry, role)
        validation_passed = True
        val_err = None
        p_status = validated["status"]
        p_answer = validated["answer"]
        v_citations = validated["citations"]
    except Exception as e:
        validation_passed = False
        val_err = str(e)
        p_status = "error"
        p_answer = ""
        v_citations = []

    results.append({
        "question_id": qid,
        "model_id": model_id,
        "raw_model_output": raw_output,
        "parsed_status": p_status,
        "parsed_answer": p_answer,
        "validated_citations": v_citations,
        "validation_passed": validation_passed,
        "validation_error": val_err,
        "generation_time_seconds": gen_time,
    })

    print(f"  Status: {p_status} | Validated: {validation_passed} | Latency: {gen_time:.2f}s")
    if p_answer:
        print(f"  Answer: {p_answer}")
    if v_citations:
        print(f"  Citations: {v_citations}")
    if val_err:
        print(f"  Validation Error: {val_err}")

In [ ]:
# Cell 7 — Export results file
import json
from google.colab import files

output_filename = "generation_results.jsonl"
with open(output_filename, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"[OK] Exported {len(results)} results to {output_filename}")
print(f"GPU Hardware Used: {gpu_name}")
print(f"Model Identifier: {model_id}")
files.download(output_filename)